In [1]:
# from pytorch documentation (https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)

In [3]:
import torch
import torch.nn as nn 
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torch.autograd import Variable
from torchvision import datasets, models, transforms
import os 
import numpy as np

In [8]:
# Data augmentation and normalization for training
# Just normalization for validation
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # this is list of mean of RGB channels, list of std. dev. of RGB channels
    ]),
}

data_dir = 'hymenoptera_data'
# creating a dictionary that contains the information of the imaged in both the traning and validation set 
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x]) for x in ['train', 'val']}
# creating a dictionary that contains the data loader 
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x],
                                              batch_size=4,
                                              shuffle=True) for x in ['train', 'val']}
# creating a dictionary that contains the size of each dataset (train + val)
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Class names : {class_names}")
print(f"There are {(len(dataloaders['train']))} batches in training set")
print(f"There are {(len(dataloaders['val']))} batches in validation set")
print(f"There are {dataset_sizes['train']} training images")
print(f"There are {dataset_sizes['val']} validation images")

# We want to be able to train our model on an `accelerator <https://pytorch.org/docs/stable/torch.html#accelerators>`__
# such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

device = torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU"
print(f"Using your {device}")

Class names : ['ants', 'bees']
There are 61 batches in training set
There are 39 batches in validation set
There are 244 training images
There are 153 validation images
Using your NVIDIA GeForce GTX 1650


In [17]:
model_conv = torchvision.models.resnet18(pretrained = True) # i can use weights = ResNet18_Weights.DEFAULT instead of pretrained argument !!
for param in model_conv.parameters() : 
    param.requires_grad = False

C:\Users\tanma\anaconda3\envs\torch_gpu\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\tanma\anaconda3\envs\torch_gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [18]:
# get the number of inputs of the last layer or equivalently no. of neurons in the penultimate layer !!
num_features = model_conv.fc.in_features
# reconstruct the last layer (output layer) to have only two classes
model_conv.fc = nn.Linear(num_features, 2)

# now move the model to the gpu ! do not do it before changing the final layers !!

if torch.cuda.is_available() : 
    model_conv = model_conv.cuda()

In [19]:
iteration = 0 
correct = 0 
for inputs, labels in dataloaders["train"] : 
    if iteration == 1 : 
        break 
    inputs = Variable(inputs)
    labels = Variable(labels)
    if torch.cuda.is_available() : 
        inputs = inputs.cuda()
        labels = labels.cuda()
    print("for one iteration this is what happens")
    print("inputs shape : ", inputs.shape)
    print("labels shape : ", labels.shape)
    print(f"labels are : {labels}")
    output = model_conv(inputs)
    print("output tensor", output)
    print("output shape : ", output.shape)
    _,preds = torch.max(output,1)
    print("Shape of Preds and preds : ",preds.shape, preds)
    correct += (preds == labels).sum()
    print("correct : ", correct)
    iteration += 1
    

for one iteration this is what happens
inputs shape :  torch.Size([4, 3, 224, 224])
labels shape :  torch.Size([4])
labels are : tensor([0, 0, 0, 1], device='cuda:0')
output tensor tensor([[-0.2725,  1.3568],
        [-0.0383, -0.0289],
        [-0.0153,  0.5734],
        [-0.0688,  0.3850]], device='cuda:0', grad_fn=<AddmmBackward0>)
output shape :  torch.Size([4, 2])
Shape of Preds and preds :  torch.Size([4]) tensor([1, 1, 1, 1], device='cuda:0')
correct :  tensor(1, device='cuda:0')


In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_conv.fc.parameters(), lr = 0.001, momentum = 0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [23]:
num_epochs = 25
for epoch in range(num_epochs) : 
    exp_lr_scheduler.step()
    correct = 0 # resetting correct for next epoch
    for images,labels in dataloaders["train"] : 
        if torch.cuda.is_available() : 
            images = Variable(images).cuda()
            labels = Variable(labels).cuda()
        optimizer.zero_grad()
        outputs = model_conv(images)
        loss = criterion(outputs,labels)
        loss.backward() # calculating grad wrt each node
        optimizer.step() # updating weights of each node 
        _,preds = torch.max(outputs,1)
        correct += (preds==labels).sum()

    train_acc = 100*correct/dataset_sizes["train"]
    print(f"Epoch {epoch+1}/{num_epochs}, Loss = {loss.item():.3f}, Train Acc. = {train_acc:.3f}")

Epoch 1/25, Loss = 0.333, Train Acc. = 75.410
Epoch 2/25, Loss = 0.014, Train Acc. = 72.951
Epoch 3/25, Loss = 0.094, Train Acc. = 78.279
Epoch 4/25, Loss = 0.462, Train Acc. = 79.098
Epoch 5/25, Loss = 0.058, Train Acc. = 83.607
Epoch 6/25, Loss = 0.201, Train Acc. = 86.066
Epoch 7/25, Loss = 0.272, Train Acc. = 83.607
Epoch 8/25, Loss = 0.017, Train Acc. = 86.475
Epoch 9/25, Loss = 0.222, Train Acc. = 81.557
Epoch 10/25, Loss = 0.435, Train Acc. = 87.295
Epoch 11/25, Loss = 0.391, Train Acc. = 86.066
Epoch 12/25, Loss = 0.290, Train Acc. = 81.148
Epoch 13/25, Loss = 0.023, Train Acc. = 81.557
Epoch 14/25, Loss = 0.079, Train Acc. = 88.115
Epoch 15/25, Loss = 0.022, Train Acc. = 81.967
Epoch 16/25, Loss = 0.081, Train Acc. = 83.607
Epoch 17/25, Loss = 0.239, Train Acc. = 84.426
Epoch 18/25, Loss = 0.191, Train Acc. = 87.705
Epoch 19/25, Loss = 0.329, Train Acc. = 87.295
Epoch 20/25, Loss = 0.457, Train Acc. = 86.885
Epoch 21/25, Loss = 0.053, Train Acc. = 83.607
Epoch 22/25, Loss = 0.

In [24]:
model_conv.eval()
with torch.no_grad() : 
    correct = 0
    total = 0
    for (images, labels) in dataloaders["val"] : 
        if torch.cuda.is_available() : 
            images = Variable(images).cuda()
            labels = Variable(labels).cuda()
        outputs = model_conv(images)
        _,preds = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    print(f"Test Acc. = {100*correct/total : .3f}")

Test Acc. =  93.464
